# 02 Adder
This notebook loads the adder onto the PYNQ board. It compares it's writes to the registers and verifies the output is correct

In [14]:
from pynq import Overlay
import random

# Load bitstream
overlay = Overlay("adder.bit")
adder = overlay.AXI_Adder_0

# Register map
REG_A = 0x00
REG_B = 0x04
REG_SUM = 0x08


def expected_add(a, b):
    """Match FPGA 32-bit adder behavior."""
    return (a + b) & 0xFFFFFFFF


def test_case(a, b):
    """Single deterministic test."""
    adder.write(REG_A, a)
    adder.write(REG_B, b)

    result = adder.read(REG_SUM)
    expected = expected_add(a, b)

    print(f"A={a}, B={b}, DUT={result}, EXPECTED={expected}")

    assert result == expected, f"Mismatch: {a} + {b}"


def run_edge_tests():
    print("\nRunning edge tests...\n")

    tests = [
        (0, 0),
        (0, 1),
        (1, 1),
        (123456, 654321),
        (0x7FFFFFFF, 1),
        (0x80000000, 1),
        (0xFFFFFFFF, 0),
        (0xFFFFFFFF, 0xFFFFFFFF),
    ]

    for a, b in tests:
        test_case(a, b)

    print("\nEdge tests passed!\n")


def run_random_tests(n=50):
    print(f"Running {n} random tests...\n")

    for _ in range(n):
        a = random.randint(0, 0xFFFFFFFF)
        b = random.randint(0, 0xFFFFFFFF)
        test_case(a, b)

    print("\nRandom tests passed!\n")


if __name__ == "__main__":
    run_edge_tests()
    run_random_tests(50)
    print("AXI Adder verification complete!")


Running edge tests...

A=0, B=0, DUT=0, EXPECTED=0
A=0, B=1, DUT=1, EXPECTED=1
A=1, B=1, DUT=2, EXPECTED=2
A=123456, B=654321, DUT=777777, EXPECTED=777777
A=2147483647, B=1, DUT=2147483648, EXPECTED=2147483648
A=2147483648, B=1, DUT=2147483649, EXPECTED=2147483649
A=4294967295, B=0, DUT=4294967295, EXPECTED=4294967295
A=4294967295, B=4294967295, DUT=4294967294, EXPECTED=4294967294

Edge tests passed!

Running 50 random tests...

A=396862534, B=3012325150, DUT=3409187684, EXPECTED=3409187684
A=4044837659, B=1412770813, DUT=1162641176, EXPECTED=1162641176
A=4103260919, B=1887877852, DUT=1696171475, EXPECTED=1696171475
A=2312095878, B=3835513976, DUT=1852642558, EXPECTED=1852642558
A=975572845, B=3671307524, DUT=351913073, EXPECTED=351913073
A=3273546180, B=8811272, DUT=3282357452, EXPECTED=3282357452
A=4268974883, B=1030365830, DUT=1004373417, EXPECTED=1004373417
A=4013679154, B=2485800253, DUT=2204512111, EXPECTED=2204512111
A=1598831909, B=3363163014, DUT=667027627, EXPECTED=667027627